In [ ]:
import os
import cv2
import numpy as np
import tifffile
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, Conv2DTranspose, Concatenate, BatchNormalization, Activation
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import LearningRateScheduler, ModelCheckpoint, EarlyStopping
from tensorflow.keras.preprocessing.image import ImageDataGenerator


In [ ]:
def dice_coef(y_true, y_pred, smooth=1):
    y_true_f = tf.keras.backend.flatten(y_true)
    y_pred_f = tf.keras.backend.flatten(y_pred)
    intersection = tf.keras.backend.sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (tf.keras.backend.sum(y_true_f) + tf.keras.backend.sum(y_pred_f) + smooth)

def iou_coef(y_true, y_pred, smooth=1):
    y_true_f = tf.keras.backend.flatten(y_true)
    y_pred_f = tf.keras.backend.flatten(y_pred)
    intersection = tf.keras.backend.sum(y_true_f * y_pred_f)
    union = tf.keras.backend.sum(y_true_f) + tf.keras.backend.sum(y_pred_f) - intersection
    return (intersection + smooth) / (union + smooth)


In [ ]:
# Function to split images into tiles
def split_image_into_tiles(image_path, mask_path, tile_size, size):
    img = tifffile.imread(image_path)
    mask = tifffile.imread(mask_path)
    mask = mask[:, :, 0] if len(mask.shape) == 3 else mask

    tiles_img, tiles_mask = [], []
    for x in range(0, img.shape[1], tile_size):
        for y in range(0, img.shape[0], tile_size):
            tile_img = img[y:y+tile_size, x:x+tile_size, :]
            tile_mask = mask[y:y+tile_size, x:x+tile_size]

            tile_img = cv2.resize(tile_img, (size, size))
            tile_mask = cv2.resize(tile_mask, (size, size))
            tile_mask = (tile_mask > 0).astype(np.uint8)

            tiles_img.append(tile_img)
            tiles_mask.append(tile_mask)

    return np.array(tiles_img), np.array(tiles_mask)

# Load dataset
def load_data(image_dir, mask_dir, tile_size=256, size=256):
    images, masks = [], []
    image_filenames = sorted(os.listdir(image_dir))
    mask_filenames = sorted(os.listdir(mask_dir))

    for image_filename in image_filenames:
        if image_filename.endswith(".TIF"):
            mask_filename = image_filename.replace(".TIF", "_mask.TIF")
            if mask_filename in mask_filenames:
                img_path = os.path.join(image_dir, image_filename)
                mask_path = os.path.join(mask_dir, mask_filename)
                img, mask = split_image_into_tiles(img_path, mask_path, tile_size, size)
                images.extend(img)
                masks.extend(mask)
    return np.array(images), np.array(masks)

# Paths
image_dir = "../datasets/images"
mask_dir = "../datasets/masks"
size = 256

# Load data
tiles_img, tiles_mask = load_data(image_dir, mask_dir, tile_size=size, size=size)

# Split sets
X_train, X_test, y_train, y_test = train_test_split(tiles_img, tiles_mask, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.1, random_state=42)

# Reshape masks
y_train = y_train[..., np.newaxis]
y_val = y_val[..., np.newaxis]
y_test = y_test[..., np.newaxis]


In [ ]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Conv2D, Conv2DTranspose, Concatenate
from tensorflow.keras.optimizers import Adam
import os
import numpy as np
import matplotlib.pyplot as plt
import cv2
import tifffile
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, BatchNormalization, Conv2DTranspose
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, LearningRateScheduler
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split



# ---- Define AlexNet-style Segmentation Model ---- #
def alexnet_segmentation_model(input_size=(256, 256, 3), freeze_encoder=True):
    inputs = Input(shape=input_size)

    # Encoder (AlexNet-style)
    x = Conv2D(96, (11, 11), strides=4, padding='same', activation='relu')(inputs)
    x = BatchNormalization()(x)
    x = MaxPooling2D((3, 3), strides=2, padding='same')(x)

    x = Conv2D(256, (5, 5), padding='same', activation='relu')(x)
    x = BatchNormalization()(x)
    x = MaxPooling2D((3, 3), strides=2, padding='same')(x)

    x = Conv2D(384, (3, 3), padding='same', activation='relu')(x)
    x = BatchNormalization()(x)

    x = Conv2D(384, (3, 3), padding='same', activation='relu')(x)
    x = BatchNormalization()(x)

    x = Conv2D(256, (3, 3), padding='same', activation='relu')(x)
    x = BatchNormalization()(x)
    x = MaxPooling2D((3, 3), strides=2, padding='same')(x)

    # Decoder (Upsampling only, no skip connections)
    x = Conv2DTranspose(256, (3, 3), strides=2, padding='same', activation='relu')(x)
    x = BatchNormalization()(x)

    x = Conv2DTranspose(128, (3, 3), strides=2, padding='same', activation='relu')(x)
    x = BatchNormalization()(x)

    x = Conv2DTranspose(64, (3, 3), strides=2, padding='same', activation='relu')(x)
    x = BatchNormalization()(x)

    x = Conv2DTranspose(32, (3, 3), strides=2, padding='same', activation='relu')(x)
    x = BatchNormalization()(x)

    output = Conv2D(1, (1, 1), activation='sigmoid')(x)

    # Resize to match ground truth
    output = tf.image.resize(output, (input_size[0], input_size[1]), method='bilinear')

    model = Model(inputs, output)
    #model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    model.compile(
    optimizer=Adam(1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy', dice_coef, iou_coef])
    #model.summary()
    return model



# Usage
size = 256
model = alexnet_segmentation_model(input_size=(size, size, 3), freeze_encoder=True)
model.summary()


In [ ]:
# LR schedule
def lr_schedule(epoch):
    initial_lr = 1e-4
    decay = 0.9
    return initial_lr * (decay ** (epoch // 10))

lr_scheduler = LearningRateScheduler(lr_schedule)

# Data augmentation
datagen = ImageDataGenerator(rescale=1./255,
                             shear_range=0.2,
                             zoom_range=0.2,
                             horizontal_flip=True,
                             rotation_range=20,
                             width_shift_range=0.2,
                             height_shift_range=0.2,
                             brightness_range=[0.8, 1.2])

# Callbacks
checkpointer = ModelCheckpoint("best_alexnet.h5", monitor="val_dice_coef", mode="max",
                               save_best_only=True, verbose=1)
earlyStopping = EarlyStopping(monitor="val_dice_coef", patience=5, mode="max", verbose=1)


In [ ]:
history = model.fit(datagen.flow(X_train, y_train, batch_size=32),
                    validation_data=(X_val/255.0, y_val),
                    epochs=50,
                    callbacks=[lr_scheduler, earlyStopping, checkpointer])


In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score
from sklearn.utils import resample



def bootstrap_confidence_interval(y_true, y_pred, metric_fn, n_bootstraps=100, alpha=0.95):
    """Bootstrap CI + std for a given metric"""
    stats = []
    n = len(y_true)
    for _ in range(n_bootstraps):
        indices = np.random.randint(0, n, n)
        if metric_fn.__name__ == "roc_auc_score":  # ROC-AUC requires probs
            stat = metric_fn(y_true[indices], y_pred[indices])
        else:  # Binary metrics
            stat = metric_fn(y_true[indices], y_pred[indices])
        stats.append(stat)
    
    stats = np.array(stats)
    mean_val = np.mean(stats)
    std_val  = np.std(stats)
    lower = np.percentile(stats, ((1 - alpha) / 2) * 100)
    upper = np.percentile(stats, (alpha + (1 - alpha) / 2) * 100)
    
    return mean_val, std_val, (lower, upper)


# --- Evaluate model ---
loss, acc, dice, iou = model.evaluate(X_test/255.0, y_test)
print(f"Test Loss: {loss:.4f}, Accuracy: {acc:.4f}, Dice: {dice:.4f}, IoU: {iou:.4f}")

# --- Predictions ---
y_pred = model.predict(X_test/255.0)
y_pred_bin = (y_pred > 0.5).astype(np.uint8)

# Flatten
y_true_flat = y_test.flatten()
y_pred_flat = y_pred_bin.flatten()
y_pred_probs = y_pred.flatten()

# --- Metrics ---
metrics = {
    "Accuracy": lambda yt, yp: np.mean(yt == yp),
    "Precision": lambda yt, yp: precision_score(yt, yp),
    "Recall": lambda yt, yp: recall_score(yt, yp),
    "F1-score": lambda yt, yp: f1_score(yt, yp),
    "ROC-AUC": lambda yt, yp: roc_auc_score(yt, yp),
    "Dice": lambda yt, yp: (2*np.sum(yt*yp))/(np.sum(yt)+np.sum(yp)+1e-7),
    "IoU": lambda yt, yp: np.sum(yt*yp)/(np.sum(yt)+np.sum(yp)-np.sum(yt*yp)+1e-7)
}

print("\n📊 Metrics with 95% Confidence Intervals:")
for name, fn in metrics.items():
    if name == "ROC-AUC":
        mean_val, std_val, (low, high) = bootstrap_confidence_interval(y_true_flat, y_pred_probs, fn)
    else:
        mean_val, std_val, (low, high) = bootstrap_confidence_interval(y_true_flat, y_pred_flat, fn)
    print(f"{name}: {mean_val:.4f} ± {std_val:.4f}  (95% CI: {low:.4f} – {high:.4f})")


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Flatten ground truth and predictions
y_true_flat = y_test.flatten()
y_pred_flat = (y_pred.flatten() > 0.5).astype(int)  # threshold at 0.5

# Confusion Matrix
cm = confusion_matrix(y_true_flat, y_pred_flat)
print("Confusion Matrix:\n", cm)

# Classification Report
print(classification_report(y_true_flat, y_pred_flat))

# Heatmap
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Non-Forest","Forest"],
            yticklabels=["Non-Forest","Forest"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()


In [ ]:
# Plot training history for loss, accuracy, dice, iou
plt.figure(figsize=(12, 6))
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')

plt.xlabel('Epoch')
plt.ylabel('Metrics')
plt.title('AlexNet Segmentation Training History')
plt.legend()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Epochs
epochs = list(range(1, 20))  # Update based on your total epochs

# Training metrics from your log
accuracy = [0.7872, 0.8489, 0.8568, 0.8561, 0.8601, 0.8617, 0.8601, 0.8627, 0.8628, 0.8642,
            0.8613, 0.8682, 0.8669, 0.8665, 0.8654, 0.8618, 0.8677, 0.8632, 0.8628]

dice_coef = [0.7182, 0.8001, 0.8094, 0.8081, 0.8103, 0.8099, 0.8158, 0.8163, 0.8148, 0.8201,
             0.8147, 0.8270, 0.8250, 0.8222, 0.8228, 0.8158, 0.8222, 0.8208, 0.8165]

iou_coef = [0.5662, 0.6686, 0.6809, 0.6797, 0.6830, 0.6833, 0.6905, 0.6912, 0.6896, 0.6969,
            0.6891, 0.7064, 0.7036, 0.6997, 0.7003, 0.6907, 0.7005, 0.6980, 0.6912]

# Validation metrics (optional, uncomment if needed)
val_accuracy = [0.7001, 0.7909, 0.7897, 0.7920, 0.7909, 0.7919, 0.7924, 0.7920, 0.7913, 0.7916,
                0.7925, 0.7917, 0.7948, 0.7918, 0.7917, 0.7922, 0.8374, 0.7917, 0.7916]

val_dice_coef = [0.5345, 0.5681, 0.6523, 0.7130, 0.7218, 0.7286, 0.7449, 0.7408, 0.7354, 0.7400,
                 0.7457, 0.7408, 0.7610, 0.7460, 0.7369, 0.7460, 0.7986, 0.7487, 0.7471]

val_iou_coef = [0.3657, 0.3978, 0.4861, 0.5565, 0.5675, 0.5757, 0.5962, 0.5911, 0.5845, 0.5902,
                0.5973, 0.5912, 0.6166, 0.5976, 0.5862, 0.5977, 0.6681, 0.6010, 0.6003]

# Plot
plt.figure(figsize=(12,6))
plt.plot(epochs, accuracy, label='Training Accuracy', marker='o')
plt.plot(epochs, dice_coef, label='Training Dice Coef', marker='x')
plt.plot(epochs, iou_coef, label='Training IoU Coef', marker='s')

plt.plot(epochs, val_accuracy, label='Validation Accuracy', marker='o', linestyle='--')
plt.plot(epochs, val_dice_coef, label='Validation Dice Coef', marker='x', linestyle='--')
plt.plot(epochs, val_iou_coef, label='Validation IoU Coef', marker='s', linestyle='--')

plt.xlabel('Epochs')
plt.ylabel('Metric Value')
plt.title('Training and Validation Metrics over Epochs')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
import matplotlib.pyplot as plt

# Epochs
epochs = list(range(1, 21))  # You shared 20 epochs

# Training metrics
train_dice = [0.7182, 0.8001, 0.8094, 0.8081, 0.8103, 0.8099, 0.8158, 0.8163, 0.8148, 0.8201,
              0.8147, 0.8270, 0.8250, 0.8222, 0.8228, 0.8158, 0.8222, 0.8208, 0.8165, 0.8184]
train_iou = [0.5662, 0.6686, 0.6809, 0.6797, 0.6830, 0.6833, 0.6905, 0.6912, 0.6896, 0.6969,
             0.6891, 0.7064, 0.7036, 0.6997, 0.7003, 0.6907, 0.7005, 0.6980, 0.6912, 0.6930]

# Validation metrics
val_dice = [0.5345, 0.5681, 0.6523, 0.7130, 0.7218, 0.7286, 0.7449, 0.7408, 0.7354, 0.7400,
            0.7457, 0.7408, 0.7610, 0.7460, 0.7369, 0.7460, 0.7986, 0.7487, 0.7411, 0.7542]
val_iou = [0.3657, 0.3978, 0.4861, 0.5565, 0.5675, 0.5757, 0.5962, 0.5911, 0.5845, 0.5902,
           0.5973, 0.5912, 0.6166, 0.5976, 0.5862, 0.5977, 0.6681, 0.6010, 0.5936, 0.6080]

# Plot Dice coefficient
plt.figure(figsize=(12,5))
plt.plot(epochs, train_dice, 'bo-', label='Train Dice')
plt.plot(epochs, val_dice, 'ro-', label='Validation Dice')
plt.xlabel('Epoch')
plt.ylabel('Dice Coefficient')
plt.title('Training vs Validation Dice Coefficient')
plt.legend()
plt.grid(True)
plt.show()

# Plot IoU
plt.figure(figsize=(12,5))
plt.plot(epochs, train_iou, 'bo-', label='Train IoU')
plt.plot(epochs, val_iou, 'ro-', label='Validation IoU')
plt.xlabel('Epoch')
plt.ylabel('IoU Coefficient')
plt.title('Training vs Validation IoU')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
import matplotlib.pyplot as plt

# Use the same data from before
epochs = range(1, 23)

# Metrics in range [0,1]
train_accuracy = [0.7872, 0.8489, 0.8568, 0.8561, 0.8601, 0.8617, 0.8601, 0.8627, 0.8628, 0.8642,
                  0.8613, 0.8682, 0.8669, 0.8665, 0.8654, 0.8618, 0.8677, 0.8632, 0.8628, 0.8668,
                  0.8657, 0.8630]

train_dice = [0.7182, 0.8001, 0.8094, 0.8081, 0.8103, 0.8099, 0.8158, 0.8163, 0.8148, 0.8201,
              0.8147, 0.8270, 0.8250, 0.8222, 0.8228, 0.8158, 0.8222, 0.8208, 0.8165, 0.8194,
              0.8231, 0.8195]

train_iou = [0.5662, 0.6686, 0.6809, 0.6797, 0.6830, 0.6833, 0.6905, 0.6912, 0.6896, 0.6969,
             0.6891, 0.7064, 0.7036, 0.6997, 0.7003, 0.6907, 0.7005, 0.6980, 0.6912, 0.6959,
             0.7013, 0.6957]

val_accuracy = [0.7001, 0.7909, 0.7897, 0.7920, 0.7909, 0.7919, 0.7924, 0.7920, 0.7913, 0.7916,
                0.7925, 0.7917, 0.7948, 0.7918, 0.7917, 0.7922, 0.8374, 0.7917, 0.7922, 0.7930,
                0.7627, 0.7919]

val_dice = [0.5345, 0.5681, 0.6523, 0.7130, 0.7218, 0.7286, 0.7449, 0.7408, 0.7354, 0.7400,
            0.7457, 0.7408, 0.7610, 0.7460, 0.7369, 0.7460, 0.7986, 0.7487, 0.7455, 0.7474,
            0.7118, 0.7352]

val_iou = [0.3657, 0.3978, 0.4861, 0.5565, 0.5675, 0.5757, 0.5962, 0.5911, 0.5845, 0.5902,
           0.5973, 0.5912, 0.6166, 0.5976, 0.5862, 0.5977, 0.6681, 0.6010, 0.5969, 0.5994,
           0.5554, 0.5841]

train_loss = [0.4453, 0.3450, 0.3331, 0.3292, 0.3289, 0.3224, 0.3202, 0.3174, 0.3168, 0.3153,
              0.3124, 0.3033, 0.3085, 0.3072, 0.3088, 0.3148, 0.3067, 0.3103, 0.3127, 0.3118,
              0.3084, 0.3102]

val_loss = [0.6634, 0.5742, 0.7813, 0.8434, 1.0205, 1.2872, 1.2462, 1.1752, 1.6788, 1.3449,
            1.0856, 1.4451, 0.6600, 1.1064, 0.9516, 0.8778, 0.3850, 1.1341, 1.1846, 0.8733,
            0.5318, 1.2864]

# Create the plot
fig, ax1 = plt.subplots(figsize=(14, 8))

# Plot metrics in [0,1] range on primary y-axis
ax1.plot(epochs, train_accuracy, 'o-', color='dodgerblue', label='Train Accuracy', linewidth=2)
ax1.plot(epochs, val_accuracy, 's--', color='dodgerblue', label='Val Accuracy', linewidth=2)

ax1.plot(epochs, train_dice, 'o-', color='green', label='Train Dice', linewidth=2)
ax1.plot(epochs, val_dice, 's--', color='green', label='Val Dice', linewidth=2)

ax1.plot(epochs, train_iou, 'o-', color='orange', label='Train IoU', linewidth=2)
ax1.plot(epochs, val_iou, 's--', color='orange', label='Val IoU', linewidth=2)

ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('Accuracy / Dice / IoU', fontsize=12, color='black')
ax1.set_title('Training and Validation Metrics Over Epochs', fontsize=14, fontweight='bold')
ax1.legend(loc='lower left', bbox_to_anchor=(0.02, 0.02), ncol=2)
ax1.grid(True, alpha=0.3)

# Create secondary y-axis for loss
ax2 = ax1.twinx()
ax2.plot(epochs, train_loss, '^-', color='red', label='Train Loss', linewidth=2, alpha=0.8)
ax2.plot(epochs, val_loss, 'v--', color='red', label='Val Loss', linewidth=2, alpha=0.8)
ax2.set_ylabel('Loss', fontsize=12, color='red')
ax2.tick_params(axis='y', labelcolor='red')

# Combine legends
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax2.legend(lines1 + lines2, labels1 + labels2, loc='center right', bbox_to_anchor=(0.98, 0.5))

# Final layout
fig.tight_layout()
plt.show()

In [ ]:
# Import necessary libraries
import matplotlib.pyplot as plt
import numpy as np

# Manually extracted metrics from your training log
# We go up to Epoch 22 (early stopping)

epochs = range(1, 23)  # 1 to 22

# Training metrics
train_loss = [
    0.4453, 0.3450, 0.3331, 0.3292, 0.3289, 0.3224, 0.3202, 0.3174, 0.3168, 0.3153,
    0.3124, 0.3033, 0.3085, 0.3072, 0.3088, 0.3148, 0.3067, 0.3103, 0.3127, 0.3118,
    0.3084, 0.3102
]

train_accuracy = [
    0.7872, 0.8489, 0.8568, 0.8561, 0.8601, 0.8617, 0.8601, 0.8627, 0.8628, 0.8642,
    0.8613, 0.8682, 0.8669, 0.8665, 0.8654, 0.8618, 0.8677, 0.8632, 0.8628, 0.8668,
    0.8657, 0.8630
]

train_dice = [
    0.7182, 0.8001, 0.8094, 0.8081, 0.8103, 0.8099, 0.8158, 0.8163, 0.8148, 0.8201,
    0.8147, 0.8270, 0.8250, 0.8222, 0.8228, 0.8158, 0.8222, 0.8208, 0.8165, 0.8194,
    0.8231, 0.8195
]

train_iou = [
    0.5662, 0.6686, 0.6809, 0.6797, 0.6830, 0.6833, 0.6905, 0.6912, 0.6896, 0.6969,
    0.6891, 0.7064, 0.7036, 0.6997, 0.7003, 0.6907, 0.7005, 0.6980, 0.6912, 0.6959,
    0.7013, 0.6957
]

# Validation metrics
val_loss = [
    0.6634, 0.5742, 0.7813, 0.8434, 1.0205, 1.2872, 1.2462, 1.1752, 1.6788, 1.3449,
    1.0856, 1.4451, 0.6600, 1.1064, 0.9516, 0.8778, 0.3850, 1.1341, 1.1846, 0.8733,
    0.5318, 1.2864
]

val_accuracy = [
    0.7001, 0.7909, 0.7897, 0.7920, 0.7909, 0.7919, 0.7924, 0.7920, 0.7913, 0.7916,
    0.7925, 0.7917, 0.7948, 0.7918, 0.7917, 0.7922, 0.8374, 0.7917, 0.7922, 0.7930,
    0.7627, 0.7919
]

val_dice = [
    0.5345, 0.5681, 0.6523, 0.7130, 0.7218, 0.7286, 0.7449, 0.7408, 0.7354, 0.7400,
    0.7457, 0.7408, 0.7610, 0.7460, 0.7369, 0.7460, 0.7986, 0.7487, 0.7455, 0.7474,
    0.7118, 0.7352
]

val_iou = [
    0.3657, 0.3978, 0.4861, 0.5565, 0.5675, 0.5757, 0.5962, 0.5911, 0.5845, 0.5902,
    0.5973, 0.5912, 0.6166, 0.5976, 0.5862, 0.5977, 0.6681, 0.6010, 0.5969, 0.5994,
    0.5554, 0.5841
]

In [ ]:
plt.figure(figsize=(16, 12))

# Plot 1: Loss
plt.subplot(2, 2, 1)
plt.plot(epochs, train_loss, 'bo-', label='Training Loss', linewidth=2)
plt.plot(epochs, val_loss, 'r-o', label='Validation Loss', linewidth=2)
plt.title('Training and Validation Loss', fontsize=14)
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, alpha=0.3)

# Plot 2: Accuracy
plt.subplot(2, 2, 2)
plt.plot(epochs, train_accuracy, 'bo-', label='Training Accuracy', linewidth=2)
plt.plot(epochs, val_accuracy, 'r-o', label='Validation Accuracy', linewidth=2)
plt.title('Training and Validation Accuracy', fontsize=14)
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True, alpha=0.3)

# Plot 3: Dice Coefficient
plt.subplot(2, 2, 3)
plt.plot(epochs, train_dice, 'bo-', label='Training Dice Coefficient', linewidth=2)
plt.plot(epochs, val_dice, 'r-o', label='Validation Dice Coefficient', linewidth=2)
plt.title('Training and Validation Dice Coefficient', fontsize=14)
plt.xlabel('Epochs')
plt.ylabel('Dice Coefficient')
plt.legend()
plt.grid(True, alpha=0.3)

# Plot 4: IoU
plt.subplot(2, 2, 4)
plt.plot(epochs, train_iou, 'bo-', label='Training IoU', linewidth=2)
plt.plot(epochs, val_iou, 'r-o', label='Validation IoU', linewidth=2)
plt.title('Training and Validation IoU', fontsize=14)
plt.xlabel('Epochs')
plt.ylabel('IoU')
plt.legend()
plt.grid(True, alpha=0.3)

# Adjust layout and show
plt.tight_layout()
plt.show()